In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import episcanpy as epi
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import h5py
import glob
import os

from pathlib import Path
from scipy import io
from helps import *

# Raw CM

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/RawFiles/spleen_lymph_206.h5ad")
adata.X=scipy.sparse.csr_matrix(adata.X, dtype="int32")
adata.layers["raw"]=adata.X.copy()
adata

In [ ]:
adata=adata[adata.obs.seurat_hash_id=="Spleen"]
adata.obs.rename({"cell_types":"CellType"}, axis=1, inplace=True)
adata

In [ ]:
adata=adata[adata.obs.batch_indices==1]
sc.pp.scrublet(adata)
adata

In [ ]:
adata.obs.rename({"cell_types":"CellType"}, axis=1, inplace=True)
adata.write("Datasets/Spleen/CM/Spleen_GEX_Raw.h5ad", compression="gzip")

In [ ]:
adt_var=pd.DataFrame(adata.uns["protein_names"], index=adata.uns["protein_names"], columns=["Protein"])
adt=sc.AnnData(X=scipy.sparse.csr_matrix(adata.obsm["protein_expression"], dtype="int32"), obs=adata.obs, var=adt_var)
adt

In [ ]:
adt.write("Datasets/Spleen/CM/Spleen_Spleen_Raw.h5ad", compression="gzip")

## mRNA & lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/CM/Spleen_GEX_Raw.h5ad")
adata.var.head()

In [ ]:
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/mm/gencode.vM33.annotation.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="HAVANA")]
gene_names=[attr.replace("gene_id", "gene_name").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_name")]
gene_names=[g.split(" ")[1][1:] for g in gene_names]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T
info.head()

In [ ]:
mRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="protein_coding"]["GeneName"]])]
print(mRNA.shape)
mRNA.write("Datasets/Spleen/CM/Spleen_mRNA_Raw.h5ad", compression="gzip")

In [ ]:
lncRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="lncRNA"]["GeneName"]])]
print(lncRNA.shape)
lncRNA.write("Datasets/Spleen/CM/Spleen_lncRNA_Raw.h5ad", compression="gzip")

# Processing

## GEX

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/CM/Spleen_GEX_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.025)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.025)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.1, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.1]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')

In [ ]:
min_var = np.quantile(adata.var.variances_norm, 0.6)
adata=adata[:, adata.var.variances_norm>min_var]
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/Spleen/CM/Spleen_GEX_Def.h5ad", compression="gzip")

## ADT

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/CM/Spleen_ADT_Raw.h5ad")
adata.var["highly_variable"]=True
adata

In [ ]:
sc.pp.normalize_total(adata)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/Spleen/CM/Spleen_ADT_Def.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/CM/Spleen_mRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
sc.pp.scrublet(adata)

In [ ]:
epi.pp.qc_stats(adata, verbose=True)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.1)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.6)
adata=adata[:, adata.var.variances_norm>min_var]

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/Spleen/CM/Spleen_mRNA_Def.h5ad", compression="gzip")

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/Spleen/CM/Spleen_lncRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.02)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.05)
adata=adata[:, adata.var.variances_norm>min_var]

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/Spleen/CM/Spleen_lncRNA_Def.h5ad", compression="gzip")

## Muon objects

In [ ]:
import muon as mu
adt=sc.read_h5ad(f"Datasets/Spleen/CM/Spleen_ADT_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/Spleen/CM/Spleen_GEX_Def.h5ad")
mrna=sc.read_h5ad(f"Datasets/Spleen/CM/Spleen_mRNA_Def.h5ad")
lncrna=sc.read_h5ad(f"Datasets/Spleen/CM/Spleen_lncRNA_Def.h5ad")

#ADT + GEX
print("ADT + GEX")
inter=intersection([adt.obs.index, gex.obs.index])
mudata=mu.MuData({"ADT" : adt[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/Spleen/CM/Spleen_ADT_GEX_Def.h5mu", compression="gzip")

#ADT + mRNA
print("ADT + mRNA")
inter=intersection([adt.obs.index, mrna.obs.index])
mudata=mu.MuData({"ADT" : adt[inter], "mRNA" : mrna[inter]})
mudata.write_h5mu(f"Datasets/Spleen/CM/Spleen_ADT_mRNA_Def.h5mu", compression="gzip")

#ADT + lncRNA
print("ADT + lncRNA")
inter=intersection([adt.obs.index, lncrna.obs.index])
mudata=mu.MuData({"ADT" : adt[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/Spleen/CM/Spleen_ADT_lncRNA_Def.h5mu", compression="gzip")

#mRNA + lncRNA
print("mRNA + lncRNA")
inter=intersection([mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/Spleen/CM/Spleen_mRNA_lncRNA_Def.h5mu", compression="gzip")

#ADT + mRNA + lncRNA
print("ADT + mRNA + lncRNA")
inter=intersection([adt.obs.index, mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"ADT" : adt[inter], "mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/Spleen/CM/Spleen_ADT_mRNA_lncRNA_Def.h5mu", compression="gzip")

# CTA

In [ ]:
df=sc.read_h5ad("Datasets/Spleen/CM/Spleen_GEX_Raw.h5ad").obs
df.head()

In [ ]:
df[["CellType"]].to_csv("Datasets/Spleen/Spleen_Metadata.tsv.gz", sep="\t", compression="gzip")